# Lab: Snowflake CoWork

📚  In this lab you will learn and practice the following:

❄️ Creating a Cortex Search Service for document search

❄️ Creating a semantic view for Cortex Analyst

❄️ Building a custom tool function for activity popularity ranking

❄️ Creating an agent using the CREATE AGENT command with multiple tools

❄️ Registering the agent with Snowflake Intelligence for use in Snowflake CoWork

❄️ Using natural language to query structured and unstructured data via the CoWork interface

❄️ Reviewing agent reasoning and planning to understand tool selection

❄️ Understanding access control and artifacts in Snowflake CoWork

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps. In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).

If you find your queries are running for more than 5 minutes, cancel and come back and try them later.

If you find models that are deprecated, use CoCo to help you fix the issue by selecting a suitable model.

---

### 🤖 Use CoCo as you go!

> **💡 TIP 1**: Use CoCo to explain complex SQL statements. Select any query and ask *"Explain this SQL"* to get a plain-language breakdown of what it does.
>
> **💡 TIP 2**: Want to learn more about any feature? Ask CoCo *"What does [feature name] do?"* to get more details and examples.
>
> **💡 TIP 3**: If you encounter a deprecated model error, ask CoCo *"Replace deprecated models in this notebook with current similar low-cost alternatives"* and it will fix them for you.

---

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.

## Introduction

Snowflake CoWork revolutionises how business users interact with data by providing **natural language interfaces** powered by advanced AI models. This lab demonstrates how to implement Snowflake CoWork for TravelBug, a travel activity booking platform, enabling business stakeholders to gain insights without technical expertise.

Snowflake CoWork uses **Cortex Agents** - AI models that can be connected to **semantic views, semantic models, Cortex Search services, and custom tools**. These agents reason through tasks, choose the right tools, and deliver results in natural language. Snowflake CoWork is powered by **Cortex Agents** which integrate **Cortex Analyst** (for structured data), **Cortex Search** (for unstructured data), and **custom tools** (for user-defined functions and procedures).

In this lab, we'll build an intelligent agent for TravelBug that can answer business questions about bookings, activities, travelers, and reviews using conversational language rather than SQL queries.

 **Business Impact of Snowflake CoWork:**
- **Democratized Data Access**: Non-technical users can now analyze TravelBug data using natural language
- **Faster Insights**: Immediate answers to business questions without waiting for reports
- **Consistent Analysis**: Standardized metrics and definitions across the organization
- **Visual Analytics**: Automatic chart and dashboard generation for trend analysis
- **Scalable Solution**: Framework that can be extended to other business domains


## **Snowflake CoWork Use Cases**

| Industry | Use Case | Business Value |
|:---|:---|:---|
| 🏥 **Healthcare** | Clinical note generation from patient conversations | Reduces admin load; improves accuracy and patient throughput |
| 💰 **Financial Services** | AI-driven insurance claims processing | Automates workflows; boosts speed and precision |
| 🛍️ **Retail & E-commerce** | Real-time AI shopping assistants | Enhances engagement; lifts conversion rates |
| 🎬 **Media & Entertainment** | Multimodal content recommendations | Increases viewer retention; reduces churn |

---

## Getting Started

Before implementing Snowflake CoWork for TravelBug, we need to understand:

* **Business Challenge**: TravelBug needs to democratize data access across the organization, allowing non-technical users to analyze booking trends, activity performance, and customer behavior

* **Common Questions**: Business users frequently ask questions like:
  - "What's our revenue by location this quarter?"
  - "Which activities have the highest booking rates?"
  - "Show me customer satisfaction trends by region"
  - "What's the average booking value by traveler segment?"

* **Business Language**: Understanding travel industry terminology (bookings, activities, travelers, reviews, conversion rates)

* **Data Quality**: Ensuring our TravelBug data in the **TRANSFORMED** schema is clean, consistent, and well-structured

* **Semantic Model**: Creating comprehensive semantic views that capture business relationships and metrics

### Setup your current context for the role, database, schema and warehouse.

In [ ]:

from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_GENAI_DB'
print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r Setup_context_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA raw;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: Snowflake CoWork';

-- Using pipe operator instead of RESULT_SCAN for better readability
SHOW PARAMETERS LIKE 'query_tag' IN SESSION 
  ->> SELECT "value" AS query_tag FROM $1;

### Load the documents for the search service.

First, let's verify we have the PDF documents that the search service will run on. The documents can be found in the **{{user}}_GENAI_DB.RESOURCES.GENAI2DAY/SEARCH_DOCS** stage.

Run the following command to confirm you can see the PDF documents.

In [ ]:
%%sql -r stage_files
LIST @{{user}}_genai_db.resources.genai2day/search_docs;

### Create the PARSE_DOC table.

Next, we create a table called **PARSE_DOC** to parse the documents using the **AI_PARSE_DOCUMENT** function. This table will include the relative path of each document, a source URL, file size, and the raw text extracted from each PDF.

In [ ]:
%%sql -r parse_doc_result
-- Create and populate PARSE_DOC in one step
CREATE TABLE IF NOT EXISTS {{user}}_genai_db.raw.PARSE_DOC AS
SELECT
    relative_path AS relative_path,
    GET_PRESIGNED_URL('@{{user}}_genai_db.resources.genai2day', relative_path) AS source_url,
    size AS size,
    TO_VARCHAR(
        AI_PARSE_DOCUMENT(
            TO_FILE('@{{user}}_genai_db.resources.genai2day', relative_path),
            {'mode': 'LAYOUT'}
        ):content
    ) AS raw_text
FROM DIRECTORY(@{{user}}_genai_db.resources.genai2day)
WHERE STARTSWITH(relative_path, 'search_docs/');

### Query the contents of the PARSE_DOC table.

Let's verify the table was populated correctly and inspect the extracted text.

In [ ]:
%%sql -r parse_doc_contents
SELECT *, LENGTH(raw_text) AS raw_text_size
FROM {{user}}_genai_db.raw.PARSE_DOC;

In [ ]:
%%sql -r Setup_Cortex_Search_create_service
CREATE OR REPLACE CORTEX SEARCH SERVICE {{user}}_GENAI_DB.RAW.TRAVELBUG_SEARCH_SERVICE
  ON raw_text
  ATTRIBUTES relative_path
  WAREHOUSE = {{user}}_GENAI_WH
  TARGET_LAG = '1 hour'
  AS (
    SELECT
      raw_text,
      relative_path
    FROM {{user}}_GENAI_DB.RAW.PARSE_DOC
  );

### Setting Up Snowflake CoWork Infrastructure (reference only - commented out intentionally)

We have pre-established the foundational infrastructure for Snowflake CoWork.

Sample commands (already executed by ACCOUNTADMIN):

```sql
--USE ROLE ACCOUNTADMIN;
--CREATE OR REPLACE SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT;
--GRANT USAGE ON SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT TO ROLE genai_role;
--GRANT MODIFY ON SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT TO ROLE genai_role;
```

> **Status**: CoWork infrastructure pre-configured by ACCOUNTADMIN

## Setup Semantic View for Cortex Analyst

The next cell contains the SQL command to create a semantic view. This command is safe to run multiple times and is designed to set up the view as needed, whether you're starting fresh or continuing from a previous setup. Rerunning it will not cause any issues.

The SQL code defines the following:

**Semantic View**: **TRAVELBUG_SEMANTIC_MODEL**

**Key Tables**: **ACTIVITY, BOOKING, REVIEW, and TRAVELER**. The view acts as a business intelligence semantic layer for these tables by integrating them.

**Relationships**: How the tables are joined.

**Measures (Measurable Facts)**: Categorizes columns into measurable facts (like price and capacity) and includes pre-calculated business metrics such as total revenue, average booking value, and the total number of reviews.

**Dimensions (Descriptive Attributes)**: e.g. **ACTIVITY_NAME** and **BOOKING_STATUS**  

This semantic view allows analysts to easily query and analyze travel data without needing to write complex join statements, enabling them to answer business questions more efficiently.

In [ ]:
%%sql -r Setup_Semantic_View_create_sql
-- Create or replace a semantic view for the Travelbug Analyst application
CREATE  SEMANTIC VIEW  IF NOT EXISTS RESOURCES.TRAVELBUG_SEMANTIC_MODEL

  -- Declare the source tables and their primary keys
  TABLES (
    ACTIVITY AS TRANSFORMED.ACTIVITY PRIMARY KEY (ACTIVITY_ID), -- Activities offered
    BOOKING AS TRANSFORMED.BOOKING PRIMARY KEY (BOOKING_ID),     -- Bookings made by travelers
    REVIEW AS TRANSFORMED.REVIEW PRIMARY KEY (REVIEW_ID),        -- Reviews submitted
    TRAVELER AS TRANSFORMED.TRAVELER PRIMARY KEY (TRAVELER_ID) -- Traveler profiles
  )

  -- Define relationships between tables for join logic
  RELATIONSHIPS (
    BOOKING (ACTIVITY_ID) REFERENCES ACTIVITY,       -- Each booking is linked to an activity
    BOOKING (TRAVELER_ID) REFERENCES TRAVELER,     -- Each booking is made by a traveler
    REVIEW (BOOKING_ID) REFERENCES BOOKING           -- Each review is tied to a booking
  )

  -- Declare measurable numeric fields (facts) for analysis
  FACTS (
    ACTIVITY.capacity AS CAPACITY                    -- Max participants per activity
        COMMENT='The maximum number of participants for an activity.',
    ACTIVITY.duration AS DURATION                    -- Duration in hours
        COMMENT='The length of time required to complete the activity, in hours.',
    ACTIVITY.price AS PRICE                          -- Price of the activity
        COMMENT='The amount paid or charged for a particular activity or service.',
    BOOKING.total_price AS TOTAL_PRICE               -- Total booking cost
        COMMENT='The total cost of a booking, including all applicable fees and taxes.'
  )

  -- Define descriptive fields (dimensions) for filtering and grouping
  DIMENSIONS (
    ACTIVITY.activity_id AS ACTIVITY_ID              -- Unique activity ID
        COMMENT='Unique identifier for a specific activity or task performed within a process or workflow.',
    ACTIVITY.activity_name AS NAME                   -- Name of the activity
        COMMENT='Name of the activity. Could be Kayaking Adventure, Rock Climbing, Sunset Boat Cruise, etc.',
    ACTIVITY.activity_description AS DESCRIPTION     -- Description of the activity
        COMMENT='A brief description of various activities offered.',
    ACTIVITY.activity_location AS LOCATION           -- Location of the activity
        COMMENT='The physical location where an activity takes place.',
    BOOKING.booking_id AS BOOKING_ID                 -- Unique booking ID
        COMMENT='Unique identifier for each booking.',
    BOOKING.booking_date AS BOOKING_DATE             -- Date the booking was made
        COMMENT='Date on which a booking was made.',
    BOOKING.activity_date AS ACTIVITY_DATE           -- Date the activity occurred
        COMMENT='Date on which a booking activity took place.',
    BOOKING.booking_status AS BOOKING_STATUS         -- Status of the booking
        COMMENT='The current status of a booking (e.g., Confirmed, Canceled).',
    BOOKING.payment_status AS PAYMENT_STATUS         -- Payment status
        COMMENT='Indicates if the payment has been successfully made (e.g., Paid, Refunded).',
    REVIEW.review_id AS REVIEW_ID                    -- Unique review ID
        COMMENT='Unique identifier for each review in the system.',
    REVIEW.review_date AS REVIEW_DATE                -- Date the review was submitted
        COMMENT='The date on which a review was submitted or posted.',
    REVIEW.review_text AS REVIEW_TEXT                -- Review content
        COMMENT='Customer reviews and feedback about their experiences.',
    TRAVELER.traveler_id AS TRAVELER_ID              -- Unique traveler ID
        COMMENT='Unique identifier for each traveler.',
    TRAVELER.traveler_name AS TRAVELER_NAME          -- Traveler name
        COMMENT='The full name of the traveler.',
    TRAVELER.traveler_email AS EMAIL                 -- Traveler email
        COMMENT='The email address of the traveler.',
    TRAVELER.traveler_address AS ADDRESS             -- Traveler address
        COMMENT='Physical address of the traveler.',
    TRAVELER.date_of_birth AS DATE_OF_BIRTH          -- Traveler birth date
        COMMENT='Date of birth of the traveler.',
    TRAVELER.phone_number AS PHONE_NUMBER            -- Traveler phone number
        COMMENT='The phone number of the traveler.'
  )

  -- Define calculated metrics using aggregate functions
  METRICS (
    BOOKING.total_revenue AS SUM(BOOKING.TOTAL_PRICE) -- Total revenue from bookings
        COMMENT='Total revenue generated from all bookings.',
    BOOKING.average_booking_value AS AVG(BOOKING.TOTAL_PRICE) -- Average booking value
        COMMENT='Average value of bookings across all transactions.',
    BOOKING.total_bookings AS COUNT(BOOKING.BOOKING_ID) -- Total number of bookings
        COMMENT='Total number of bookings made.',
    ACTIVITY.average_duration AS AVG(ACTIVITY.DURATION) -- Average activity duration
        COMMENT='Average duration of activities in hours.',
    ACTIVITY.average_capacity AS AVG(ACTIVITY.CAPACITY) -- Average activity capacity
        COMMENT='Average capacity across all activities.',
    ACTIVITY.average_price AS AVG(ACTIVITY.PRICE)       -- Average activity price
        COMMENT='Average price of activities.',
    ACTIVITY.activity_count AS COUNT(ACTIVITY.ACTIVITY_ID) -- Total number of activities
        COMMENT='Total number of unique activities available.',
    REVIEW.total_reviews AS COUNT(REVIEW.REVIEW_ID)     -- Total number of reviews
        COMMENT='Total number of reviews submitted.',
    TRAVELER.traveler_count AS COUNT(DISTINCT TRAVELER.TRAVELER_ID) -- Unique travelers
        COMMENT='Total number of unique travelers.'
  )

-- Add a description for the semantic view
COMMENT='This semantic model for the Travelbug Analyst application focuses on the traveler, booking, activity, and review tables in the transformed schema. It is designed to help answer business questions raised by analysts and includes comprehensive metrics for performance analysis.'
;


## Setup Custom Tools for Snowflake CoWork

Before setting up the Snowflake CoWork infrastructure, we'll create custom tools that can be called by our agents. Custom tools allow agents to execute stored procedures and functions, extending capabilities beyond standard analytics.

For TravelBug, we'll create an **Activity Popularity Ranker** that provides real-time popularity rankings based on recent booking data.

### Create the activity popularity ranking function.

This function provides popularity rankings for TravelBug activities based on booking data.
Since we do not have real-time data, we run this function over all the booking data.

Key Steps:

**Calculate Core Metrics**: The code first creates a temporary table (activity_rankings) that calculates key performance indicators for every activity. It counts total confirmed bookings, finds the average booking value, and counts the number of unique customers.

**Rank All Activities**: Using window functions (RANK()), it then assigns two separate ranks to each activity within this temporary table:

**Popularity Rank**: Based on the total number of bookings (more bookings = higher rank).

**Value Rank**: Based on the average booking price (higher value = higher rank).

**Find the Target**: The function filters this comprehensive ranked list to find the single row that matches the input_activity_name provided when the function is called.

**Construct the Summary**: Finally, it takes all the calculated numbers for that specific activity (its ranks, booking counts, etc.) and uses CONCAT to assemble them into a single, easy-to-read text string, which is the function's final output.

In [ ]:
%%sql -r Create_activity_popularity_ranking_sql
-- Set the schema to 'RESOURCES' for the current session.
USE SCHEMA RESOURCES;

-- Create or replace a SQL User-Defined Function (UDF) to get a comprehensive popularity ranking for a given activity.
CREATE OR REPLACE FUNCTION get_activity_popularity_rank(input_activity_name VARCHAR)
-- The function returns a single string (VARCHAR) summarizing the activity's rank and stats.
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
    -- Start a Common Table Expression (CTE) named 'activity_rankings' to calculate metrics for all activities.
    WITH activity_rankings AS (
        SELECT
            -- Select the activity name.
            a.NAME as activity_name,
            -- Count the number of confirmed bookings for each activity.
            COUNT(b.booking_id) as recent_bookings,
            -- Calculate the average booking price, rounded to two decimal places. Use COALESCE to handle NULL.
            COALESCE(ROUND(AVG(b.total_price), 2), 0) as avg_booking_value,
            -- Count the number of unique travelers who have booked the activity.
            COUNT(DISTINCT b.traveler_id) as unique_customers,
            -- Assign a popularity rank based on the number of bookings (descending). The most booked activity is #1.
            RANK() OVER (ORDER BY COUNT(b.booking_id) DESC) as popularity_rank,
            -- Assign a value rank based on the average booking value (descending). The highest average value is #1.
            RANK() OVER (ORDER BY COALESCE(ROUND(AVG(b.total_price), 2), 0) DESC) as value_rank
        -- Start with the 'activity' table as the base.
        FROM {{user}}_genai_db.TRANSFORMED.activity a
        -- Join with the 'booking' table on the activity_id to link activities to their bookings.
        LEFT JOIN {{user}}_genai_db.TRANSFORMED.booking b ON a.activity_id = b.activity_id
        -- Filter bookings to include only those with a 'Confirmed' status.
          AND b.booking_status = 'Confirmed'
        -- The following line is commented out but could be enabled to restrict the analysis to bookings made in the last 30 days.
        --  AND b.booking_date >= DATEADD(days, -30, CURRENT_DATE())
        -- Group the results by activity name to aggregate the metrics for each unique activity.
        GROUP BY a.NAME
    ),
    -- Create a second CTE to get the total count of distinct activities.
    total_activities AS (
        SELECT COUNT(*) as total_count FROM activity_rankings
    )
    -- Final SELECT statement to construct the output string.
    SELECT CONCAT(
        'Popularity Rank: #', popularity_rank, ' out of ', ta.total_count, ' activities. ',
        'Recent bookings: ', recent_bookings, '. ',
        'Unique customers: ', unique_customers, '. ',
        'Average booking value: ', avg_booking_value, '. ',
        'Value Rank: #', value_rank, '.'
    )
    -- Select from both CTEs. The cross join is intentional as 'total_activities' has only one row.
    FROM activity_rankings ar, total_activities ta
    -- Filter the results to find the specific activity passed into the function. The comparison is case-insensitive.
    WHERE UPPER(ar.activity_name) = UPPER(input_activity_name)
$$;

### Check the current activity names.

Run the query below to check the unique activity names.

In [ ]:
%%sql -r Check_the_current_activity_names_sql

-- First, let's check what activities exist in our data
SELECT DISTINCT NAME as activity_name
FROM {{user}}_genai_db.TRANSFORMED.activity 
ORDER BY NAME;


### Test the ranking function.

We call the function with an activity name to see it's ranking. 

In [ ]:
%%sql -r Test_the_ranking_function_sql
-- Test the function with a sample activity
SELECT get_activity_popularity_rank('Mountain Hiking') AS popularity_analysis;


In [ ]:
df = Test_the_ranking_function_sql.to_pandas()

run_output = df["POPULARITY_ANALYSIS"].iloc[0]
print(run_output)

In [ ]:
%%sql -r Setting_Up_Snowflake_Intelligence_sql
USE ROLE genai_role;

-- Verify agent database and schema
SHOW SCHEMAS IN DATABASE {{user}}_genai_db;


## Creating the TravelBug CoWork Agent

Now, we'll create an intelligent agent specifically designed for TravelBug business analytics. We can create an agent using the Snowsight UI, SQL or REST API. In this case we will use the **CREATE AGENT** SQL command. 

You can add your agent to SNOWFLAKE INTELLIGENCE OBJECT using ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT ADD AGENT <db.schema.agent_name>;

Run the following SQL to create the TravelBug CoWork Agent with all tools configured:

In [ ]:
%%sql -r Creating_the_TravelBug_Intelligence_sql
-- Create the TravelBug CoWork Agent
CREATE OR REPLACE AGENT {{user}}_AGENT
  COMMENT = 'TravelBug CoWork Agent with Cortex Analyst, Cortex Search, and custom tools'
  PROFILE = '{"display_name": "{{user}} AGENT"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 300
    tokens: 16000

instructions:
  response: |
    You are a business intelligence assistant for TravelBug, a travel booking platform. Always:
    - Provide clear, business-focused answers
    - Create visualizations when appropriate (charts, graphs)
    - Use travel industry terminology correctly
    - Round financial figures to 2 decimal places
    - Explain trends and insights in business context
    - Offer actionable recommendations when relevant
  sample_questions:
    - question: "What is the total revenue?"
    - question: "How popular was the hot air balloon activity last year?"

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "{{user}}_TRAVELBUG_ANALYTICS"
      description: "TravelBug semantic view containing booking, activity, traveler, and review data with comprehensive business metrics for travel industry analytics."

  - tool_spec:
      type: "cortex_search"
      name: "{{user}}_TRAVELBUG_DOC_ANALYSIS"
      description: "TravelBug search service to extract information from unstructured documents."

  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Generates visualizations from data"

  - tool_spec:
      type: "generic"
      name: "{{user}}_ACTIVITY_POPULARITY_RANKER"
      description: "Get popularity rankings and performance metrics for TravelBug activities based on booking data."
      input_schema:
        type: "object"
        properties:
          input_activity_name:
            type: "string"
            description: "The name of the activity to get popularity ranking for (e.g., Kayaking Adventure, Rock Climbing, Hot Air Balloon)"
        required:
          - input_activity_name

tool_resources:
  {{user}}_TRAVELBUG_ANALYTICS:
    semantic_view: "{{user}}_GENAI_DB.RESOURCES.TRAVELBUG_SEMANTIC_MODEL"
    execution_environment:
      type: "warehouse"
      warehouse: "{{user}}_GENAI_WH"

  {{user}}_TRAVELBUG_DOC_ANALYSIS:
    name: "{{user}}_GENAI_DB.RESOURCES.TRAVELBUG_SEARCH"
    id_column: "source_url"
    title_column: "relative_path"
    max_results: "5"

  {{user}}_ACTIVITY_POPULARITY_RANKER:
    type: "function"
    identifier: "{{user}}_GENAI_DB.RESOURCES.GET_ACTIVITY_POPULARITY_RANK"
    execution_environment:
      type: "warehouse"
      warehouse: "{{user}}_GENAI_WH"
      query_timeout: 300
$$;

In [ ]:
%%sql -r register_agent_result
-- Register the agent with Snowflake Intelligence so it can be used via the CoWork UI.
-- If the agent is already registered, this will succeed without error.
BEGIN
    ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT ADD AGENT {{user}}_genai_db.resources.{{user}}_AGENT;
EXCEPTION
    WHEN OTHER THEN
        RETURN 'Agent is already registered with Snowflake Intelligence. No action needed.';
END;

## Testing the TravelBug CoWork Agent

Now we'll test our agent with various business questions to ensure it's working correctly. Remember to duplicate your browser before you continue.

### Access Snowflake CoWork interface.

**Step 1: Navigate to Snowflake CoWork**
1. From the left-hand navigation, select **AI & ML**
2. Select **Snowflake CoWork**
3. The interface will switch to focus on Snowflake CoWork
4. Click on your animal name bottom left in Snowflake CoWork Screen
5. Click on **Settings**. Ensure the **Role** is set to **GENAI_ROLE**

 💡 Tip: Open Snowflake CoWork in a new browser tab for the best experience.

**Step 2: Select Your Agent**
1. Below the chat input, click the agent selector button and choose **{{user}}_AGENT**
2. You're now ready to start asking questions!
3. The agent will automatically determine which tools to use (Cortex Analyst, Cortex Search, or custom tools) based on your question

### Test with sample questions.



Try these questions in the Snowflake CoWork interface:

**Revenue Analysis Questions:**
1. "What's our total revenue for this year?"
2. "Show me revenue breakdown by location"
3. "What's the average booking value?"
4. "Compare revenue by quarter"
5. "Which locations generate the highest revenue?"

**Activity Performance Questions:**
1. "What are our most popular activities?"
2. "Show me activity booking trends"
3. "Which activities have the highest capacity utilization?"
4. "What's the average duration of our activities?"
5. "Compare activity prices by location"

**Customer Insights Questions:**
1. "How many unique travelers do we have?"
2. "What's the booking frequency per traveler?"
3. "Show me traveler demographics"
4. "Which travelers are our highest value customers?"

**Review Analysis Questions:**
1. "What's our overall review rate?"
2. "Show me review trends over time"
3. "Which activities get the most reviews?"
4. "How does review rate vary by location?"

**Combining Structured And Unstructured Data**
1. How well is the wine tour doing? Is the wine tour guide good?
  2. Analyze the tour portfolio, tour feedback, generate insights for Travelbug

### Testing Custom Tools in Snowflake CoWork.

Now we'll test our custom Activity Popularity Ranker tool along with the other agent capabilities.

Try these questions to test the custom tool functionality:

**Activity Popularity Analysis:**
1. "What's the popularity ranking of the Sunset Boat Cruise?"
2. "How popular is the Rock Climbing activity?"
3. "Can you tell me about the Wine Tour's popularity and performance?"
4. "Show me the ranking and recent performance of Kayaking Adventure"
5. "Which activity performs better - Mountain Biking or Hot Air Balloon?"

**Combined Analysis Questions:**
1. "Get the popularity ranking for the Wine Tour and also search for customer feedback about wine tours"
2. "Show me the Rock Climbing popularity data and find any safety information from our documents"
3. "Compare the popularity of water activities like Kayaking and Boat Cruise"

 💡 **Tip**: When asking about activity popularity, the agent will use the custom **get_activity_popularity_rank** function to provide real-time data including:
- Current popularity ranking
- Booking performance over time
- Number of unique customers
- Average booking value
- Value ranking by revenue

The agent can combine this data with other tools (Cortex Analyst and Cortex Search) to provide comprehensive insights.

## Agent Reasoning and Planning

It is possible for us to see how the agent arrived at a particular response - the process it went through for reasoning, the tools it used, and the queries it ran.

### Looking at reasoning and planning.

Click on **Show Details** after a question and see the steps the agent has followed. You should see output similar to the following based on your question.

**Planning the next steps**
The user is asking about the ranking and recent performance of "Wine Tours". This seems to be asking about a specific activity in the TravelBug system. I should use the ACTIVITY_POPULARITY_RANKER tool to get information about Wine Tours' ranking and performance based on recent booking data.

*   **Running ACTIVITY_POPULARITY_RANKER**
*   **Executing tool ACTIVITY_POPULARITY_RANKER**
*   **Planning the next steps**

.
.

## Access Control for Snowflake CoWork

Snowflake CoWork follows a caller's-rights model - all queries run under the user's own credentials, so existing RBAC and data-masking policies automatically apply. Users need `USAGE` privileges on the agent and its tools (Cortex Search services, semantic views, custom tools). Access is controlled via the `SNOWFLAKE.CORTEX_USER` or the more selective `SNOWFLAKE.CORTEX_AGENT_USER` database role.

## Saving and Sharing Artifacts

When Snowflake CoWork generates a chart or table that contains a useful insight, you can save it as an **artifact**. An artifact preserves the query, visualization, and context so you can revisit it later with refreshed data, or share it with a teammate via a link. Shared artifacts follow a caller's-rights model - each viewer sees results filtered through their own role-based access controls. You can manage all your saved and shared artifacts from the **Artifacts Hub** in the Snowflake CoWork interface.

## Beyond This Lab: The Full Agent Tool Ecosystem

This lab demonstrates three core tools (Cortex Analyst, Cortex Search, and Custom Tools). In production, Cortex Agents support a broader set of tool types that you can combine within a single agent:

| Tool | Description |
|:---|:---|
| **Cortex Analyst** | Generates SQL queries over structured data from natural language, using a semantic view |
| **Cortex Search** | Retrieves information from unstructured data with dynamic filters, columns, and time-decay settings |
| **Code Execution** | Runs Python in a secure, isolated sandbox to process data and perform calculations |
| **Data to Chart** | Generates visualizations from data returned by other tools |
| **Custom Tools** | Stored procedures and UDFs that implement your own business logic or call backend systems |
| **Agent Skills** | Packaged, modular bundles of instructions and scripts that give an agent repeatable, task-specific capabilities |
| **MCP Connectors** | Tools hosted on remote Model Context Protocol (MCP) servers, such as Atlassian Jira, Salesforce, or your own applications |
| **Web Search** | Real-time information from the public internet (must be enabled at the account level) |

All tools are declared in the agent specification YAML under the `tools` and `tool_resources` sections. The agent's orchestrator dynamically selects which tools to invoke based on the user's question.

For full details, see: [Cortex Agents Documentation](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents)

## 🎯 Challenge Questions

Test your understanding of Snowflake CoWork concepts covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "Where can agents be created to use with Snowflake CoWork?", "options": ["A) Only in USER_DB", "B) Only in SNOWFLAKE_INTELLIGENCE database", "C) Only in CORTEX_AGENTS database", "D) Any database, then added to the Snowflake Intelligence object"], "hash": "77f6f4cda9d8d56a2644dd0d67f7df75", "correct": None},
    {"q": "What do MCP connectors enable for Cortex Agents?", "options": ["A) Faster SQL query execution", "B) Direct Python library imports", "C) Connection to remote tool servers like Jira or Salesforce", "D) Automatic model fine-tuning"], "hash": None, "correct": "C"},
    {"q": "How does a Cortex Agent determine which tool to use for a given question?", "options": ["A) The user must manually specify the tool", "B) Tools are selected in a fixed order", "C) The orchestrator dynamically selects tools based on the user's query", "D) Each tool runs in parallel on every question"], "hash": None, "correct": "C"},
    {"q": "What is a key benefit of combining multiple tools in a Snowflake CoWork agent?", "options": ["A) Faster query execution", "B) Lower compute costs", "C) Simplified security model", "D) Generate insights from both structured and unstructured data sources"], "hash": "f10ca22d058d942b3645f359c17d900a", "correct": None},
    {"q": "Why should answers from Snowflake CoWork agents be verified?", "options": ["A) The agents are still in beta", "B) To verify accuracy before making business decisions", "C) The models have limited knowledge", "D) Verification is required by Snowflake"], "hash": "313648afd8b46aa703d8a305a68c8c03", "correct": None},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        if item["hash"] is not None:
            escaped_opt = opt.replace("'", "''")
            result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
            feedback = result[0][0]
            is_correct = 'Correct' in feedback or '\u2705' in feedback
        else:
            is_correct = (letter == item["correct"])
            feedback = "\u2705 Correct" if is_correct else "\u274c Incorrect. Try again"
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))

## Key Takeaways

❄️ **Snowflake CoWork** provides a conversational AI interface where agents answer business questions using structured and unstructured data.

❄️ **CREATE AGENT** defines an agent with tools (Cortex Analyst, Cortex Search, custom functions) and registers it with `ALTER SNOWFLAKE INTELLIGENCE ... ADD AGENT`.

❄️ **Cortex Analyst** (text-to-SQL via semantic views) and **Cortex Search** (document retrieval) can be combined in a single agent for comprehensive insights.

❄️ **Custom tools** extend agent capabilities by calling stored procedures/functions, enabling real-time business logic like popularity rankings.

❄️ **Caller's-rights model**: all queries run under the user's credentials, so RBAC and data-masking policies automatically apply.

❄️ **Artifacts**: save charts and tables generated by the agent for later review or sharing with teammates via links.

❄️ **Agent reasoning** is transparent. Click "Show Details" to see which tools were selected, what queries were run, and how the answer was assembled.